# 26 — Gold Point-in-Time Audit

**Objetivo:** demostrar que el dataset final de modelado puede reconstruirse como si estuviéramos ubicados en cada `t0`, sin información futura.

Este notebook materializa `data/gold/ml/argos_v1.parquet` únicamente después de pasar las auditorías de grain, labels, leakage y timestamps.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.pit_audit import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

In [ ]:
from argos.gold.pit_audit import build_pit_audit

result = build_pit_audit(
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

str(result.artifact.relative_to(paths.root))

## Feature registry auditado

In [ ]:
result.feature_registry

In [ ]:
result.validation

In [ ]:
result.issue_register

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("26 ✅ PASS" if result.ready_for_gate else "26 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_gate, "Gold PIT audit permanece OPEN; no ejecutar gate M3."